Load in data

In [ ]:
# imports

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd 
import numpy as np
import time

# use the widget backend for interactive plots inside the notebook
# make sure `ipympl` is installed and the kernel has been restarted
%matplotlib widget

# TRY NORMALIZING DATA
# StandardScalarx

In [ ]:

proper = pd.read_csv('proper_asteroid_data_small.csv')
families = pd.read_csv('family_membership.csv')

df = proper.merge(families[['name','family1']], on='name', how='left')
df['family1'] = df['family1'].fillna(0).astype(int)

family_names = {
    4:    'Vesta',
    15:   'Eunomia',
    20:   'Massalia',
    24:   'Themis',
    158:  'Koronis',
    170:  'Maria',
    221:  'Eos',
    490:  'Veritas',
    847:  'Agnia',
    1040: 'Natasha',
}
feature_columns = ['a', 'e', 'sin_i']
X = df[feature_columns].dropna().values
len(df)

Look for most common asteroid families
Eos
Eunomia?
Flora
Hungaria
Hygiea
Koronis
Nysa
Themis
Vesta


In [ ]:
def plot_family(family_id):
    name = family_names.get(family_id, f'Family {family_id}')
    members = df[df['family1'] == family_id]
    background = df[df['family1'] != family_id]
    print(f"{name} members in dataset: {len(members)}")
    a_min, a_max = members['a'].min() - 0.3, members['a'].max() + 0.3

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    plots = [('a','e'), ('a','sin_i'), ('e','sin_i')]

    for ax, (xcol, ycol) in zip(axes, plots):
        ax.scatter(background[xcol], background[ycol], s=1, c='lightgray', alpha=0.3, label='background')
        ax.scatter(members[xcol], members[ycol], s=8, c='red', alpha=0.7, label=f'{name} (n={len(members)})')
        ax.set_xlabel(xcol)
        ax.set_ylabel(ycol)
        ax.set_title(f'{name} — {xcol} vs {ycol}')
        ax.legend()
        # Zoom in based on members' data
        x_min = members[xcol].min() - 0.1
        x_max = members[xcol].max() + 0.1
        y_min = members[ycol].min() - 0.1
        y_max = members[ycol].max() + 0.1
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)

    plt.tight_layout()
    plt.savefig(f'{name.lower()}_family.png', dpi=150)
    plt.show()

In [ ]:
# plot_family(4)


In [ ]:
from mpl_toolkits.mplot3d import Axes3D    # noqa: F401 – needed for the 3‑D backend

def plot_family_3d(family_id):
    name = family_names.get(family_id, f'Family {family_id}')
    members    = df[df['family1'] == family_id]
    background = df[df['family1'] != family_id]

    print(f"{name} members in dataset: {len(members)}")

    fig = plt.figure(figsize=(8, 6))
    ax  = fig.add_subplot(111, projection='3d')

    ax.scatter(background['a'], background['e'], background['sin_i'],
               s=1, c='lightgray', alpha=0.3, label='background')
    ax.scatter(members['a'], members['e'], members['sin_i'],
               s=8, c='red', alpha=0.7, label=f'{name} (n={len(members)})')

    ax.set_xlabel('a')
    ax.set_ylabel('e')
    ax.set_zlabel('sin_i')
    ax.set_title(f'{name} family in 3‑D orbital element space')
    ax.legend()
    plt.tight_layout()
    plt.show()

# call it for Vesta (family 4)
# plot_family_3d(221)

In [ ]:
# plot_family(490)

In [ ]:
# Run this cell FIRST before anything else

ZONE_BOUNDARIES = {
    2: (2.065, 2.3),
    3: (2.3,   2.501),
    4: (2.501, 2.825),
    5: (2.825, 2.958),
    6: (2.958, 3.030),
    7: (3.030, 3.278),
}

# NEW cutoffs based on sweep analysis
ZONE_CUTOFFS = {
    2: 60,
    3: 60,
    4: 80,
    5: 80,
    6: 80,
    7: 120,
}

In [ ]:
# from scipy.cluster.hierarchy import linkage, fcluster
# from scipy.spatial.distance import pdist
# k1, k2, k3 = 5/4, 2, 2 # paper section 2
# zone_boundaries = [(0.0, 2.065), (2.065, 2.3), (2.3, 2.501), (2.501, 2.825), (2.825, 2.958), (2.958, 3.030), (3.030, 3.278), (3.278, 4)] # dont know where it ends
# zone_cutoff = (0, 60, 30, 50, 20, 30, 120, 0) # 1 and 8 dont have a boundary
# def compute_delta_v_matrix_chunked(X, chunk_size=1000, verbose=True):
#     """
#     Compute all pairwise delta_v distances in chunks to stay within memory.
    
#     Instead of computing all N*(N-1)/2 pairs at once (which needs ~20GB for
#     32k asteroids), we process chunk_size rows at a time. Each chunk computes
#     distances from rows [i : i+chunk_size] to all rows j > i, then moves on.
#     Peak memory is proportional to chunk_size * N, not N^2.
    
#     X          : array (N, 3) of [a', e', sin_i']
#     chunk_size : number of rows to process at once. 
#                  1000 → ~250MB peak, 500 → ~125MB peak
#     Returns    : condensed distance array, length N*(N-1)/2
#     """
#     N = len(X)
#     n_pairs = N * (N - 1) // 2
#     dist_condensed = np.empty(n_pairs, dtype=np.float32)  # float32 halves memory vs float64

#     a   = X[:, 0]
#     e   = X[:, 1]
#     si  = X[:, 2]

#     pair_idx = 0  # position in dist_condensed we're writing to

#     for i_start in range(0, N, chunk_size):
#         i_end = min(i_start + chunk_size, N)

#         if verbose and i_start % (chunk_size * 20) == 0:
#             pct = 100 * pair_idx / n_pairs
#             print(f"    {pct:.1f}% ({pair_idx:,} / {n_pairs:,} pairs)", flush=True)

#         for i in range(i_start, i_end):
#             # Compute distances from asteroid i to all asteroids j > i
#             j_start = i + 1
#             if j_start >= N:
#                 continue

#             a_i  = a[i]
#             e_i  = e[i]
#             si_i = si[i]

#             # All j > i at once — shape (N - j_start,)
#             a_j  = a[j_start:]
#             e_j  = e[j_start:]
#             si_j = si[j_start:]

#             a0  = (a_i + a_j) / 2.0
#             n0  = 2.0 * np.pi / (a0 ** 1.5)
#             da  = a_i - a_j
#             de  = e_i - e_j
#             dsi = si_i - si_j

#             dv = n0 * a0 * np.sqrt(
#                 k1 * (da / a0)**2 +
#                 k2 * de**2 +
#                 k3 * dsi**2
#             ) * 4740.9

#             n_new = len(dv)
#             dist_condensed[pair_idx : pair_idx + n_new] = dv
#             pair_idx += n_new

#     return dist_condensed


# def hcm(df, zone, min_members=5, save_linkage=True, chunk_size=1000):
#     a_min, a_max = zone_boundaries[zone]
#     cutoff_ms    = zone_cutoff[zone]

#     zone_df = df[
#         (df['a'] >= a_min) & (df['a'] < a_max)
#     ].dropna(subset=['a', 'e', 'sin_i']).copy()

#     n = len(zone_df)
#     n_pairs = n * (n - 1) // 2
#     mem_mb  = n_pairs * 4 / 1e6  # float32 = 4 bytes
#     print(f"Zone {zone} ({a_min}–{a_max} AU): {n:,} asteroids | "
#           f"{n_pairs:,} pairs | ~{mem_mb:.0f} MB for dist array")

#     X = zone_df[['a', 'e', 'sin_i']].values

#     print(f"  Computing distances (chunk_size={chunk_size})...")
#     dist_condensed = compute_delta_v_matrix_chunked(X, chunk_size=chunk_size)

#     print(f"  Building dendrogram...")
#     Z = linkage(dist_condensed, method='single')

#     if save_linkage:
#         np.save(f'Z_zone{zone}.npy', Z)
#         zone_df.to_csv(f'zone{zone}_df.csv', index=False)
#         print(f"  Saved Z_zone{zone}.npy and zone{zone}_df.csv")

#     labels  = fcluster(Z, t=cutoff_ms, criterion='distance')
#     zone_df = zone_df.copy()
#     zone_df['hcm_label'] = labels

#     families = []
#     for label_id in np.unique(labels):
#         members = zone_df[zone_df['hcm_label'] == label_id]['name'].tolist()
#         if len(members) >= min_members:
#             families.append(members)
#     families.sort(key=len, reverse=True)

#     print(f"  → {len(families)} families found")
#     return families, zone_df

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
def compute_delta_v_matrix_chunked(X, chunk_size=1000, verbose=True):
    """
    Compute all pairwise delta_v distances in chunks to stay within memory.
    
    Instead of computing all N*(N-1)/2 pairs at once (which needs ~20GB for
    32k asteroids), we process chunk_size rows at a time. Each chunk computes
    distances from rows [i : i+chunk_size] to all rows j > i, then moves on.
    Peak memory is proportional to chunk_size * N, not N^2.
    
    X          : array (N, 3) of [a', e', sin_i']
    chunk_size : number of rows to process at once. 
                 1000 → ~250MB peak, 500 → ~125MB peak
    Returns    : condensed distance array, length N*(N-1)/2
    """
    k1, k2, k3 = 5/4, 2, 2 # paper section 2

    N = len(X)
    n_pairs = N * (N - 1) // 2
    dist_condensed = np.empty(n_pairs, dtype=np.float32)  # float32 halves memory vs float64

    a   = X[:, 0]
    e   = X[:, 1]
    si  = X[:, 2]

    pair_idx = 0  # position in dist_condensed we're writing to

    for i_start in range(0, N, chunk_size):
        i_end = min(i_start + chunk_size, N)

        if verbose and i_start % (chunk_size * 20) == 0:
            pct = 100 * pair_idx / n_pairs
            print(f"    {pct:.1f}% ({pair_idx:,} / {n_pairs:,} pairs)", flush=True)

        for i in range(i_start, i_end):
            # Compute distances from asteroid i to all asteroids j > i
            j_start = i + 1
            if j_start >= N:
                continue

            a_i  = a[i]
            e_i  = e[i]
            si_i = si[i]

            # All j > i at once — shape (N - j_start,)
            a_j  = a[j_start:]
            e_j  = e[j_start:]
            si_j = si[j_start:]

            a0  = (a_i + a_j) / 2.0
            n0  = 2.0 * np.pi / (a0 ** 1.5)
            da  = a_i - a_j
            de  = e_i - e_j
            dsi = si_i - si_j

            dv = n0 * a0 * np.sqrt(
                k1 * (da / a0)**2 +
                k2 * de**2 +
                k3 * dsi**2
            ) * 4740.9

            n_new = len(dv)
            dist_condensed[pair_idx : pair_idx + n_new] = dv
            pair_idx += n_new

    return dist_condensed


def hcm(df, zone, min_members=5, save_linkage=True, chunk_size=1000):
    a_min, a_max = ZONE_BOUNDARIES[zone]
    cutoff_ms    = ZONE_CUTOFFS[zone]

    zone_df = df[
        (df['a'] >= a_min) & (df['a'] < a_max)
    ].dropna(subset=['a', 'e', 'sin_i']).copy()

    n = len(zone_df)
    n_pairs = n * (n - 1) // 2
    mem_mb  = n_pairs * 4 / 1e6  # float32 = 4 bytes
    print(f"Zone {zone} ({a_min}–{a_max} AU): {n:,} asteroids | "
          f"{n_pairs:,} pairs | ~{mem_mb:.0f} MB for dist array")

    X = zone_df[['a', 'e', 'sin_i']].values

    print(f"  Computing distances (chunk_size={chunk_size})...")
    dist_condensed = compute_delta_v_matrix_chunked(X, chunk_size=chunk_size)

    print(f"  Building dendrogram...")
    Z = linkage(dist_condensed, method='single')

    if save_linkage:
        np.save(f'Z_zone{zone}.npy', Z)
        zone_df.to_csv(f'zone{zone}_df.csv', index=False)
        print(f"  Saved Z_zone{zone}.npy and zone{zone}_df.csv")

    labels  = fcluster(Z, t=cutoff_ms, criterion='distance')
    zone_df = zone_df.copy()
    zone_df['hcm_label'] = labels

    families = []
    for label_id in np.unique(labels):
        members = zone_df[zone_df['hcm_label'] == label_id]['name'].tolist()
        if len(members) >= min_members:
            families.append(members)
    families.sort(key=len, reverse=True)

    print(f"  → {len(families)} families found")
    return families, zone_df

In [ ]:
import psutil

ram_gb = psutil.virtual_memory().available / 1e9
print(f"Available RAM: {ram_gb:.1f} GB")
# fam, dffff = hcm(df, 3, 5)
# # Check each zone's memory requirement

for zone, (a_min, a_max) in ZONE_BOUNDARIES.items():
    n = len(df[(df['a'] >= a_min) & (df['a'] < a_max)])
    n_pairs = n * (n-1) // 2
    mb = n_pairs * 4 / 1e6  # float32
    print(f"Zone {zone}: {n:>6,} asteroids → {mb:>8,.0f} MB for dist array")

In [ ]:
import time

def run_all_zones(df, min_members=5):
    all_families = []
    total_start = time.time()

    for zone in ZONE_BOUNDARIES:
        t0 = time.time()
        fams, _ = hcm(df, zone, min_members)
        all_families.extend(fams)
        print(f"  Zone {zone} done in {time.time()-t0:.1f}s\n")

    print(f"All zones done in {time.time()-total_start:.1f}s")
    print(f"Total: {len(all_families)} families, "
          f"{sum(len(f) for f in all_families):,} asteroids assigned")
    return all_families

all_families = run_all_zones(df)

In [ ]:
def evaluate_completeness(all_families, df, family_names, threshold=0.95):
    """
    For each known family, find the best-matching HCM family and report
    completeness (recall) = |recovered ∩ true| / |true|
    
    A family 'passes' if completeness >= threshold.
    """
    # Build name → found-family-index lookup
    name_to_found = {}
    for i, fam in enumerate(all_families):
        for name in fam:
            name_to_found[name] = i

    rows = []
    for fam_id, fam_name in family_names.items():
        true_members = set(df[df['family1'] == fam_id]['name'].tolist())
        if not true_members:
            print(f"  {fam_name}: no members found in df, skipping")
            continue

        # Count how many true members landed in each HCM family
        vote = {}
        for name in true_members:
            if name in name_to_found:
                fi = name_to_found[name]
                vote[fi] = vote.get(fi, 0) + 1

        if not vote:
            completeness    = 0.0
            recovered       = 0
            found_size      = 0
        else:
            best_fi         = max(vote, key=vote.get)
            recovered       = vote[best_fi]
            completeness    = recovered / len(true_members)
            found_size      = len(all_families[best_fi])

        passes = completeness >= threshold
        rows.append({
            'family':       fam_name,
            'true_size':    len(true_members),
            'recovered':    recovered,
            'found_size':   found_size,
            'completeness': round(completeness * 100, 1),
            'pass':         '✓' if passes else '✗',
        })

    results = pd.DataFrame(rows).sort_values('completeness', ascending=False)
    
    # Pretty print
    print(f"{'':2}{'Family':12} {'True':>6} {'Recov':>6} "
          f"{'Found':>6} {'Compl':>7}  Pass")
    print("-" * 50)
    for _, r in results.iterrows():
        print(f"  {r['family']:12} {r['true_size']:>6} {r['recovered']:>6} "
              f"{r['found_size']:>6} {r['completeness']:>6.1f}%  {r['pass']}")
    
    n_pass = results['pass'].eq('✓').sum()
    print(f"\n{n_pass}/{len(results)} families reach {int(threshold*100)}% completeness")
    return results

results = evaluate_completeness(all_families, df, family_names)

In [ ]:
def evaluate_completeness(all_families, df, family_names, threshold=0.95):
    name_to_found = {}
    for i, fam in enumerate(all_families):
        for name in fam:
            name_to_found[name] = i

    rows = []
    for fam_id, fam_name in family_names.items():
        true_members = set(df[df['family1'] == fam_id]['name'].tolist())
        if not true_members:
            continue

        vote = {}
        for name in true_members:
            if name in name_to_found:
                fi = name_to_found[name]
                vote[fi] = vote.get(fi, 0) + 1

        if not vote:
            completeness = precision = f1 = 0.0
            recovered = found_size = 0
        else:
            best_fi      = max(vote, key=vote.get)
            recovered    = vote[best_fi]
            found_size   = len(all_families[best_fi])
            completeness = recovered / len(true_members)      # recall
            precision    = recovered / found_size             # what fraction of found are true members
            f1           = (2 * precision * completeness /
                           (precision + completeness)
                           if (precision + completeness) > 0 else 0.0)

        passes = completeness >= threshold
        rows.append({
            'family':       fam_name,
            'true_size':    len(true_members),
            'recovered':    recovered,
            'found_size':   found_size,
            'completeness': round(completeness * 100, 1),
            'precision':    round(precision * 100, 1),
            'f1':           round(f1 * 100, 1),
            'pass':         '✓' if passes else '✗',
        })

    results = pd.DataFrame(rows).sort_values('completeness', ascending=False)

    print(f"{'':2}{'Family':12} {'True':>6} {'Recov':>6} {'Found':>7} "
          f"{'Compl':>7} {'Prec':>7} {'F1':>7}  Pass")
    print("-" * 70)
    for _, r in results.iterrows():
        print(f"  {r['family']:12} {r['true_size']:>6} {r['recovered']:>6} "
              f"{r['found_size']:>7} {r['completeness']:>6.1f}% "
              f"{r['precision']:>6.1f}% {r['f1']:>6.1f}%  {r['pass']}")

    n_pass = results['pass'].eq('✓').sum()
    print(f"\n{n_pass}/{len(results)} families reach {int(threshold*100)}% completeness")
    return results

In [ ]:
def sweep_cutoff(zone, cutoffs, min_members=5):
    """
    Load a saved Z matrix and try multiple cutoffs.
    Shows how the largest family and number of families changes.
    Helps you find the right cutoff without rerunning HCM.
    """
    Z       = np.load(f'Z_zone{zone}.npy')
    zone_df = pd.read_csv(f'zone{zone}_df.csv')
    n_total = len(zone_df)

    print(f"Zone {zone} — {n_total:,} asteroids total")
    print(f"{'Cutoff':>8} {'N_families':>12} {'Largest':>10} "
          f"{'2nd_largest':>12} {'Assigned%':>10}")
    print("-" * 60)

    results = []
    for c in cutoffs:
        labels   = fcluster(Z, t=c, criterion='distance')
        unique, counts = np.unique(labels, return_counts=True)
        big      = counts[counts >= min_members]
        big_sorted = np.sort(big)[::-1]

        largest     = big_sorted[0] if len(big_sorted) > 0 else 0
        second      = big_sorted[1] if len(big_sorted) > 1 else 0
        n_fam       = len(big_sorted)
        assigned    = big_sorted.sum() / n_total * 100

        print(f"{c:>8} {n_fam:>12,} {largest:>10,} {second:>12,} {assigned:>9.1f}%")
        results.append((c, n_fam, largest, second, assigned))

    return results

# Try a wide range for zone 4 (where Eunomia/Maria/Agnia all merged)
sweep_cutoff(zone=4, cutoffs=[20, 30, 40, 50, 60, 80, 100, 120, 160])

In [ ]:
# Sweep all zones that have problem families
for zone in [2, 3, 4, 5, 6, 7]:
    print()
    sweep_cutoff(zone=zone, cutoffs=[20, 30, 40, 50, 60, 80, 100, 120, 160])

In [ ]:
def recut_all_zones(cutoffs, df, min_members=5):
    all_families = []
    for zone, cutoff in cutoffs.items():
        Z       = np.load(f'Z_zone{zone}.npy')
        zone_df = pd.read_csv(f'zone{zone}_df.csv')

        labels  = fcluster(Z, t=cutoff, criterion='distance')
        zone_df['hcm_label'] = labels

        families = []
        for label_id in np.unique(labels):
            members = zone_df[zone_df['hcm_label'] == label_id]['name'].tolist()
            if len(members) >= min_members:
                families.append(members)
        families.sort(key=len, reverse=True)

        largest = len(families[0]) if families else 0
        print(f"Zone {zone} @ {cutoff} m/s → {len(families)} families, "
              f"largest={largest:,}")
        all_families.extend(families)

    print(f"\nTotal: {len(all_families)} families, "
          f"{sum(len(f) for f in all_families):,} asteroids assigned")
    return all_families

# Apply new cutoffs using saved Z matrices — should be instant
all_families = recut_all_zones(ZONE_CUTOFFS, df)
results = evaluate_completeness(all_families, df, family_names)

In [ ]:
def sweep_with_truth(zone, cutoffs, df, family_names, min_members=5):
    """
    Sweep cutoffs for a zone and show recovery of known families at each level.
    Much more informative than just looking at cluster sizes.
    """
    Z       = np.load(f'Z_zone{zone}.npy')
    zone_df = pd.read_csv(f'zone{zone}_df.csv')

    # Which known families live in this zone?
    a_min, a_max = ZONE_BOUNDARIES[zone]
    zone_family_ids = []
    for fam_id, fam_name in family_names.items():
        members_in_zone = df[
            (df['family1'] == fam_id) &
            (df['a'] >= a_min) & (df['a'] < a_max)
        ]
        if len(members_in_zone) > 0:
            zone_family_ids.append((fam_id, fam_name, len(members_in_zone)))

    if not zone_family_ids:
        print(f"Zone {zone}: no known families")
        return

    print(f"\nZone {zone} ({a_min}–{a_max} AU)")
    print(f"Known families in this zone: "
          f"{', '.join(f'{n}({c})' for _,n,c in zone_family_ids)}")
    print()

    # Header
    header = f"{'Cutoff':>8} {'Largest':>8}"
    for _, fname, ftrue in zone_family_ids:
        header += f"  {fname[:8]:>10}({ftrue})"
    print(header)
    print("-" * len(header))

    for c in cutoffs:
        labels = fcluster(Z, t=c, criterion='distance')
        zone_df['hcm_label'] = labels

        # For each known family, find best matching cluster
        row = f"{c:>8} "

        # largest cluster size
        _, counts = np.unique(labels, return_counts=True)
        row += f"{counts.max():>8}"

        for fam_id, fname, ftrue in zone_family_ids:
            # merge ground truth into zone_df
            true_names = set(df[df['family1'] == fam_id]['name'].tolist())

            # vote for best cluster
            vote = {}
            for name in true_names:
                match = zone_df[zone_df['name'] == name]
                if len(match) > 0:
                    lbl = match['hcm_label'].iloc[0]
                    vote[lbl] = vote.get(lbl, 0) + 1

            if vote:
                best_lbl     = max(vote, key=vote.get)
                recovered    = vote[best_lbl]
                cluster_size = (zone_df['hcm_label'] == best_lbl).sum()
                pct          = recovered / ftrue * 100
                row += f"  {recovered:>4}/{ftrue:<4} {pct:>4.0f}% (sz={cluster_size})"
            else:
                row += f"  {'—':>14}"

        print(row)

In [ ]:
def plot_family_vs_hcm(family_id, d_cutoff=60):
    name = family_names.get(family_id, f'Family {family_id}')

    # Ground truth members
    members = df[df['family1'] == family_id]

    # Find seed and run HCM
    seed_idx = df[df['name'] == family_id].index[0]
    hcm_indices = hcm(X, seed_idx, d_cutoff=d_cutoff)
    hcm_mask = np.array(list(hcm_indices))
    hcm_members = df.iloc[hcm_mask]

    print(f"{name} — AstDys: {len(members)}, HCM: {len(hcm_indices)}")

    plots = [('a', 'e'), ('a', 'sin_i'), ('e', 'sin_i')]
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))

    for col, (xcol, ycol) in enumerate(plots):
        x_min = members[xcol].min() - 0.1
        x_max = members[xcol].max() + 0.1
        y_min = members[ycol].min() - 0.1
        y_max = members[ycol].max() + 0.1

        # Top row — ground truth
        ax = axes[0, col]
        ax.scatter(df[xcol], df[ycol], s=1, c='lightgray', alpha=0.3, label='background')
        ax.scatter(members[xcol], members[ycol], s=8, c='red', alpha=0.7,
                   label=f'AstDys (n={len(members)})')
        ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)
        ax.set_xlabel(xcol); ax.set_ylabel(ycol)
        ax.set_title(f'{name} Ground Truth — {xcol} vs {ycol}')
        ax.legend()

        # Bottom row — HCM result
        ax = axes[1, col]
        ax.scatter(df[xcol], df[ycol], s=1, c='lightgray', alpha=0.3, label='background')
        ax.scatter(hcm_members[xcol], hcm_members[ycol], s=8, c='blue', alpha=0.7,
                   label=f'HCM (n={len(hcm_indices)})')
        ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)
        ax.set_xlabel(xcol); ax.set_ylabel(ycol)
        ax.set_title(f'{name} HCM (cutoff={d_cutoff} m/s) — {xcol} vs {ycol}')
        ax.legend()

    plt.tight_layout()
    plt.savefig(f'{name.lower()}_hcm_vs_truth.png', dpi=150)
    plt.show()

# Usage
# plot_family_vs_hcm(4)   # Vesta

In [ ]:
all_families = recut_all_zones(ZONE_CUTOFFS, df)
results = evaluate_completeness(all_families, df, family_names)

In [ ]:
# Zone 4: Eunomia, Maria, Agnia
sweep_with_truth(4, [30, 40, 50, 60, 70, 80, 100, 120, 160], df, family_names)

# Zone 2: Massalia  
sweep_with_truth(2, [30, 40, 50, 60, 70, 80, 100, 120, 160], df, family_names)

# Zone 5: Koronis
sweep_with_truth(5, [20, 30, 40, 50, 60, 80, 100, 120, 160], df, family_names)

# Zone 6: Eos
sweep_with_truth(6, [30, 40, 50, 60, 80, 100, 120, 160], df, family_names)

# Zone 3: Vesta
sweep_with_truth(3, [20, 30, 40, 50, 60, 80, 100], df, family_names)

In [ ]:
def check_merge(zone, cutoff, df, fam_id_1, fam_id_2):
    """
    Check whether two known families ended up in the same HCM cluster
    at a given cutoff, or are still separate.
    """
    Z       = np.load(f'Z_zone{zone}.npy')
    zone_df = pd.read_csv(f'zone{zone}_df.csv')
    labels  = fcluster(Z, t=cutoff, criterion='distance')
    zone_df['hcm_label'] = labels

    def best_cluster(fam_id):
        true_names = set(df[df['family1'] == fam_id]['name'].tolist())
        vote = {}
        for name in true_names:
            match = zone_df[zone_df['name'] == name]
            if len(match) > 0:
                lbl = match['hcm_label'].iloc[0]
                vote[lbl] = vote.get(lbl, 0) + 1
        if not vote:
            return None, 0
        best = max(vote, key=vote.get)
        return best, vote[best]

    lbl1, rec1 = best_cluster(fam_id_1)
    lbl2, rec2 = best_cluster(fam_id_2)

    name1 = family_names[fam_id_1]
    name2 = family_names[fam_id_2]

    print(f"Zone {zone} @ {cutoff} m/s:")
    print(f"  {name1}: cluster {lbl1}, recovered {rec1}")
    print(f"  {name2}: cluster {lbl2}, recovered {rec2}")

    if lbl1 == lbl2:
        print(f"  ⚠ MERGED into same cluster {lbl1}")
    else:
        print(f"  ✓ Still separate clusters")

# Check the critical zones
check_merge(3, 60, df, 4, 20)    # Vesta vs Massalia at cutoff=60
check_merge(3, 50, df, 4, 20)    # same at 50

check_merge(4, 80, df, 15, 170)  # Eunomia vs Maria at cutoff=80
check_merge(4, 80, df, 15, 847)  # Eunomia vs Agnia at cutoff=80
check_merge(4, 80, df, 170, 847) # Maria vs Agnia at cutoff=80

In [ ]:
vesta_region = df[(df['a'] > 2.1) & (df['a'] < 2.6)].copy().reset_index(drop=True)
X_small = vesta_region[['a', 'e', 'sin_i']].values

print(f"Vesta region size: {len(X_small)}")  # should be ~50-100k, much more manageable

vesta_idx_small = vesta_region[vesta_region['name'] == 4].index[0]
fam = hcm(X_small, vesta_idx_small, d_cutoff=60)
print(f"Family size: {len(fam)}")

In [ ]:
# Check ground truth count in your regional subset
vesta_true = vesta_region[vesta_region['family1'] == 4]
print(f"AstDys Vesta members in this region: {len(vesta_true)}")
print(f"Total asteroids in region: {len(vesta_region)}")

for cutoff in [30, 40, 50, 60]:
    t = time.time()
    result = hcm(X, vesta_idx_small, d_cutoff=cutoff)
    print(f"cutoff={cutoff:3d} m/s -> HCM={len(result):6d} | "
          f"AstDys={len(vesta_true):6d} | "
          f"ratio={len(result)/max(len(vesta_true),1):.2f} | "
          f"{time.time()-t:.1f}s")

In [ ]:
print(f"X shape: {X.shape}")
print(f"X_test shape: {X_small.shape}")
print(f"vesta_region shape: {vesta_region.shape}")
print(f"AstDys Vesta members in vesta_region: {len(vesta_true)}")
print(f"vesta_idx_small: {vesta_idx_small}")

In [ ]:
result = hcm(X_small, vesta_idx_small, d_cutoff=42)

for cutoff in [41, 42, 43, 44, 45, 46, 47, 48, 49]:
    result = hcm(X_small, vesta_idx_small, d_cutoff=cutoff)
    print(f"cutoff={cutoff} -> HCM={len(result)} | AstDys=3410 | ratio={len(result)/3410:.2f}")

In [ ]:
# What families are in the vesta_region?
family_counts = vesta_region[vesta_region['family1'] > 0]['family1'].value_counts()
print("Families in vesta_region:")
print(family_counts.head(15))

# Plot the region colored by family
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 6))

# background
bg = vesta_region[vesta_region['family1'] == 0]
ax.scatter(bg['a'], bg['e'], s=1, c='lightgray', alpha=0.2)

# top 5 families in region, each a different color
colors = ['red','blue','green','orange','purple']
for (fid, count), color in zip(family_counts.head(5).items(), colors):
    fam = vesta_region[vesta_region['family1'] == fid]
    name = family_names.get(fid, str(fid))
    ax.scatter(fam['a'], fam['e'], s=4, c=color, 
               alpha=0.7, label=f'{name} (n={count})')

ax.set_xlabel('a (AU)'); ax.set_ylabel('e')
ax.legend(); ax.set_title('All families in Vesta region')
plt.tight_layout()
plt.savefig('vesta_region_families.png', dpi=150)
plt.show()

In [ ]:
AU_m = 1.496e11        # meters per AU
yr_s = 365.25*24*3600  # seconds per year
print(f"1 AU/yr = {AU_m/yr_s:.2f} m/s")  # ~4740 m/s